In [1]:
import os
import base64
import requests
import pandas as pd
from urllib.parse import urlparse
from dotenv import load_dotenv
from time import sleep

# === Load GitHub Token ===
load_dotenv("All_Tokens.env")
# Use a single specific token for now
token = os.getenv("GITHUB_TOKEN_6")
if not token:
    raise ValueError("GITHUB_TOKEN_6 not found in the .env file.")

# # Uncomment these lines for token rotation in future
# tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
# tokens = [t for t in tokens if t]
# if not tokens:
#     raise ValueError("No GitHub tokens found in the .env file.")
# token_index = 0

def get_headers():
    return {
        "Authorization": f"token {token}",  # static token
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-checker"
    }

# # Uncomment for future token rotation
# def rotate_token():
#     global token_index
#     token_index = (token_index + 1) % len(tokens)

# === Extract Repo Info ===
def extract_repo_info(url):
    parts = urlparse(url).path.strip("/").split("/")
    return parts[0], parts[1] if len(parts) >= 2 else (None, None)

# === Load Input CSV ===
input_path = "D:/Android_Mobile_App/AndroidProject_dataset/Added by OR.csv"
df = pd.read_csv(input_path)

results = []

# === Analyze Each Repo ===
for idx, url in enumerate(df['html_url'], start=1):
    print(f"[{idx}/{len(df)}] Checking repo: {url}")
    owner, repo = extract_repo_info(url)
    if not owner or not repo:
        continue

    base_url = f"https://api.github.com/repos/{owner}/{repo}"

    try:
        # --- Repo Metadata ---
        r = requests.get(base_url, headers=get_headers())
        if r.status_code == 403:
            # rotate_token()
            sleep(2)
            r = requests.get(base_url, headers=get_headers())
        repo_data = r.json()

        # --- Repo Topics ---
        topic_url = f"{base_url}/topics"
        r = requests.get(topic_url, headers={**get_headers(), "Accept": "application/vnd.github.mercy-preview+json"})
        if r.status_code == 403:
            # rotate_token()
            sleep(2)
            r = requests.get(topic_url, headers={**get_headers(), "Accept": "application/vnd.github.mercy-preview+json"})
        topics = r.json().get("names", [])

        # --- README Content ---
        readme_url = f"{base_url}/readme"
        r = requests.get(readme_url, headers=get_headers())
        if r.status_code == 403:
            # rotate_token()
            sleep(2)
            r = requests.get(readme_url, headers=get_headers())

        readme_text = ""
        if r.status_code == 200:
            content = r.json().get("content", "")
            readme_text = base64.b64decode(content).decode("utf-8", errors="ignore")

        # --- Checks ---
        android_in_topic = "android" in topics
        android_in_name_or_desc = "android" in f"{repo_data.get('name', '')} {repo_data.get('description', '')}".lower()
        android_in_readme = "android" in readme_text.lower()
        language = repo_data.get("language", "")

        is_android_repo = android_in_topic or android_in_name_or_desc or android_in_readme

        results.append({
            "repo_url": url,
            "is_android_repo": is_android_repo,
            "language": language,
            "android_in_topic": android_in_topic,
            "android_in_name_or_desc": android_in_name_or_desc,
            "android_in_readme": android_in_readme
        })

    except Exception as e:
        results.append({
            "repo_url": url,
            "error": str(e)
        })

# === Save Output ===
output_df = pd.DataFrame(results)
output_path = "D:/Android_Mobile_App/AndroidProject_dataset/Added_Repos_byOR_review.csv"
output_df.to_csv(output_path, index=False)

print(f"✅ Review complete. Output saved to:\n{output_path}")


[1/2484] Checking repo: https://github.com/0xZhangKe/NotionLight
[2/2484] Checking repo: https://github.com/103style/SpeedControl
[3/2484] Checking repo: https://github.com/10miaomiao/bili-down-out
[4/2484] Checking repo: https://github.com/1139618418/WaveView
[5/2484] Checking repo: https://github.com/12-10-8/ncnn-android-pose
[6/2484] Checking repo: https://github.com/121880399/LargeImageMonitor
[7/2484] Checking repo: https://github.com/121880399/QuickMvp
[8/2484] Checking repo: https://github.com/13217300237/HookSkinDemoFromHank
[9/2484] Checking repo: https://github.com/15829238397/CN5E-shop
[10/2484] Checking repo: https://github.com/1754048656/FATJS
[11/2484] Checking repo: https://github.com/18Gray/CommonUtils
[12/2484] Checking repo: https://github.com/18Gray/ProCamera
[13/2484] Checking repo: https://github.com/1998lixin/WeChat-database
[14/2484] Checking repo: https://github.com/19snow93/ShopCartLogic
[15/2484] Checking repo: https://github.com/1nikolas/play-integrity-checke